In [1]:
import torch

from loss_config import LossConfig
from DG_learn_loss import build_dg_learn_loss

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

device: cpu


/home/jeongyuseong/anaconda3/envs/pretext_task_gen/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
cfg = LossConfig()

# AE loss
cfg.distance.alpha_r = 1.0
cfg.distance.alpha_p = 1.0

# Degradation loss
cfg.distance.beta_r = 1.0
cfg.distance.beta_p = 1.0
cfg.distance.beta_m = 0.1

# Regulation
cfg.regluation.lambda_deg = 0.5
cfg.regluation.lambda_tv = 1.0
cfg.regluation.lambda_sparsity = 1.0

cfg.regluation.lambda_rep = 0.2
cfg.regluation.rep_threshold = 0.3
cfg.regluation.rep_exp_k = 10.0

ae_loss_fn, dg_loss_fn = build_dg_learn_loss(cfg)

In [3]:
B = 4
patch_num = 14
patch_size = 16
patch_dim = 3 * patch_size * patch_size
n_dim = 256

# image
x = torch.randn(B, 3, 224, 224, device=device)

# degradation output
x_dg = x + 0.1 * torch.randn_like(x)

# encoder outputs
z_x = torch.randn(B, n_dim, device=device)
z_x_dg = torch.randn(B, n_dim, device=device)

# decoder output (MAE 스타일)
pred = torch.randn(B, patch_num * patch_num, patch_dim, device=device)

In [4]:
ae_loss, ae_detail = ae_loss_fn(
    pred=pred,
    z_x=z_x,
    z_x_dg=z_x_dg,
    x=x,
)

print("AE Loss:", ae_loss.item())

for k, v in ae_detail.items():
    print(k, v.mean().item())

ValueError: x and x_dg are required for degradation distance

In [ ]:
dg_loss, dg_detail = dg_loss_fn(
    pred=pred,
    z_x=z_x,
    z_x_dg=z_x_dg,
    x=x,
    x_dg=x_dg,
)

print("DG Loss:", dg_loss.item())

for k, v in dg_detail.items():
    print(k, v.mean().item())

## 6. Gradient 흐름 테스트 (Cell 6) ⭐ 매우 중요

In [ ]:
# requires_grad 활성화
pred.requires_grad_(True)
z_x.requires_grad_(True)
z_x_dg.requires_grad_(True)
x_dg.requires_grad_(True)

# AE step
ae_loss, _ = ae_loss_fn(
    pred=pred,
    z_x=z_x,
    z_x_dg=z_x_dg.detach(),   # AE는 M 고정
    x=x,
)

ae_loss.backward()

print("AE gradients:")
print("pred grad:", pred.grad.abs().mean().item())
print("z_x grad:", z_x.grad.abs().mean().item())
print("z_x_dg grad:", z_x_dg.grad)  # None이어야 정상

## 7. Degradation gradient 테스트 (Cell 7)

In [ ]:
# grad 초기화
pred.grad = None
z_x.grad = None
z_x_dg.grad = None
x_dg.grad = None

dg_loss, _ = dg_loss_fn(
    pred=pred.detach(),   # AE 고정
    z_x=z_x.detach(),
    z_x_dg=z_x_dg,
    x=x,
    x_dg=x_dg,
)

dg_loss.backward()

print("DG gradients:")
print("z_x_dg grad:", z_x_dg.grad.abs().mean().item())
print("x_dg grad:", x_dg.grad.abs().mean().item())
print("z_x grad:", z_x.grad)   # None이어야 정상

In [ ]:
8. Loss scale sanity 체크 (Cell 8)

In [ ]:
print("=== SANITY CHECK ===")
print("AE loss:", ae_loss.item())
print("DG loss:", dg_loss.item())

assert not torch.isnan(ae_loss), "AE loss NaN 발생"
assert not torch.isnan(dg_loss), "DG loss NaN 발생"

assert ae_loss.item() < 100, "AE loss 너무 큼"
assert dg_loss.item() < 100, "DG loss 너무 큼"

print("PASS")

## 9. 하이퍼 파라미터 OFF 테스트 (Cell 9)

In [ ]:
cfg.distance.alpha_r = 0.0
cfg.distance.alpha_p = 1.0

ae_loss_fn, dg_loss_fn = build_dg_learn_loss(cfg)

ae_loss, detail = ae_loss_fn(
    pred=pred,
    z_x=z_x,
    z_x_dg=z_x_dg,
    x=x,
)

print(detail.keys())  # d_rep 없어야 정상